# Get preds on every hyper-param combination across folds/repeats

## Set Up

In [ ]:
import os
import sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH
from src.utils import ALPHA_LIST, L1_LIST
from src.swarm import run_swarms, make_full_lr, make_full_shot

In [ ]:
SEG_TYPES = [
    "turns",
    "pt_noshort",
]
MODEL_NAMES = [
    "Llama-3.3-8B-Instruct",
    "Llama-3.3-70B-Instruct",
]
IMPACT_TYPES = [
    "self",
    "parent",
]
PROB_AGG_TYPES = [
    "block",
    "max",
    "avg",
    "proba-limited",
    "binary",
]
DESC_TYPES = [
    "original",
    "augment",
    "augment_neg",
]
LINEAR_TYPES = ["elastic", "ridge"]

## Get model preds at every:

- (Self, Parent) x Alpha Val X L1 Val x Repeat (1-10) x test fold combination for REDUCED
- (Self, Parent) x (block, avg, max) x (8B, 70B) x (turns, pt_noshort) x Alpha Val X L1 Val x Repeat (1-10) x test fold combination for FULL

In [ ]:
## Note that split_tol, n_splits, n_repeats set as globals in swarm.py
N_JOBS = 4
N_THREADS = 2
TUNE_FLAG = "True"
SWARM_DIR = BASE_PATH / "swarm_dir" / "base"
metric_agg = "combined_folds"  # combined_folds or per_fold

In [ ]:
cmd_dir = SWARM_DIR / "commands"
full_cmd_list = []
for agg_type in PROB_AGG_TYPES:
    for impact_type in IMPACT_TYPES:
        swarm_path = cmd_dir / agg_type / f"{impact_type}.swarm"
        swarm_path.parent.mkdir(exist_ok=True, parents=True)
        if swarm_path.exists():
            swarm_path.unlink()
        cmd_list = []
        for linear_type in LINEAR_TYPES:
            if linear_type == "ridge":
                alpha_list = [0.1, 0.5, 1.0, 2.5, 5.0, 7.5, 10.0]
            else:
                alpha_list = ALPHA_LIST
            for n_layers in [2, 3]:
                BASE_SAVE_DIR = (
                    BASE_PATH / f"initial_preds" / f"{n_layers}_layers" / linear_type
                )
                ## Full model (block, max, avg)
                for model_name in MODEL_NAMES:
                    if agg_type in ["block", "max", "avg"]:
                        lr_cmd_list = make_full_lr(
                            seg_types=SEG_TYPES,
                            model_name=model_name,
                            impact_type=impact_type,
                            agg_type=agg_type,
                            base_save_dir=BASE_SAVE_DIR,
                            n_jobs=N_JOBS,
                            alpha_list=alpha_list,
                            l1_list=L1_LIST,
                            n_threads=N_THREADS,
                            tune_flag_str=TUNE_FLAG,
                            n_layers=n_layers,
                            linear_type=linear_type,
                            metric_agg=metric_agg,
                        )
                        cmd_list += lr_cmd_list
                    ## 0 shot
                    else:
                        for desc_type in DESC_TYPES:
                            cmd_str = make_full_shot(
                                model_name=model_name,
                                impact_type=impact_type,
                                temp=0.6,
                                pred_type=agg_type,  # one of binary, proba-limited
                                desc_type=desc_type,
                                base_save_dir=BASE_SAVE_DIR,
                                n_jobs=N_JOBS,
                                alpha_list=alpha_list,
                                l1_list=L1_LIST,
                                n_threads=N_THREADS,
                                tune_flag_str=TUNE_FLAG,
                                n_layers=n_layers,
                                linear_type=linear_type,
                                metric_agg=metric_agg,
                            )
                            cmd_list.append(cmd_str)
                swarm_path.write_text("\n".join(cmd_list))
                full_cmd_list += cmd_list
print(f"Num commands: {len(full_cmd_list)}")

Run sequentially locally

In [ ]:
# import subprocess

# BASE_SAVE_DIR.mkdir(exist_ok=True, parents=True)
# for iteration, cmd in enumerate(full_cmd_list):
#     print(f"{iteration +1}/{len(full_cmd_list)}...")
#     print(cmd)
#     subprocess.run(cmd, shell=True, check=True)
#     break

Run swarm

In [ ]:
run_swarms(
    agg_list=PROB_AGG_TYPES,
    impact_list=["parent", "self"],
    swarm_log_dir=SWARM_DIR / "logs",
    cmd_dir=cmd_dir,
    n_jobs=N_JOBS,
    n_threads=N_THREADS,
)